In [0]:
dbutils.widgets.removeAll()

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql import functions as F

In [0]:
dbutils.widgets.text("catalogo", "catalog_pry")
dbutils.widgets.text("esquema_source", "bronze")
dbutils.widgets.text("esquema_sink", "silver")

In [0]:
catalogo = dbutils.widgets.get("catalogo")
esquema_source = dbutils.widgets.get("esquema_source")
esquema_sink = dbutils.widgets.get("esquema_sink")

In [0]:
df_customers = spark.table(f"{catalogo}.{esquema_source}.customers")
df_products = spark.table(f"{catalogo}.{esquema_source}.products")
df_sales = spark.table(f"{catalogo}.{esquema_source}.sales")
print('Dataset cargados para aplicar join')

In [0]:
df_orders_enriched = df_sales.join(
    df_customers, df_sales["CustomerID"] == df_customers["CustomerID"], "inner"
).join(
    df_products, df_sales["ProductID"] == df_products["ProductID"], "inner"
).select(
    df_sales["SalesID"].cast("integer"),
    df_sales["CustomerID"].cast("integer"),
    concat_ws(" ", df_customers["FirstName"], df_customers["LastName"]).cast("string").alias("CustomerName"),
    df_customers["Address"].cast("string").alias("CustomerEmail"),
    df_sales["ProductID"].cast("integer"),
    df_products["ProductName"].cast("string"),
    df_products["Class"].cast("string").alias("Category"),
    df_sales["Quantity"].cast("integer"),
    df_products["Price"].cast("double"),
    (df_sales["Quantity"] * df_products["Price"]).cast("double").alias("TotalValue"),
    df_sales["SalesDate"].cast("timestamp")
)
display(df_orders_enriched)

In [0]:
df_orders_enriched.count()

In [0]:
df_orders_enriched_final = df_orders_enriched.withColumn("ingestion_date", current_timestamp())

In [0]:
df_orders_enriched_final.write.mode("overwrite").insertInto(f"{catalogo}.{esquema_sink}.orders_enriched")